# Fine-tuning DeepSeek-R1 (Distill Llama 8B) with Unsloth and Ollama

This notebook fine-tunes the distilled Llama variant of DeepSeek-R1 with LoRA adapters using
[Unsloth](https://unsloth.ai/blog/deepseek-r1), exports the result to GGUF and serves it with
[Ollama](https://ollama.com).

**Requirements:** an NVIDIA GPU runtime (a free Google Colab T4 is enough). It does not run on
Apple Silicon: train on Colab, then download the GGUF to use it locally (see the last section).

Steps:
1. Install dependencies
2. Load the 4-bit base model and attach LoRA adapters
3. Prepare the dataset
4. Train
5. Export to GGUF and create an Ollama model
6. Chat with the fine-tuned model

### 1. Install

In [ ]:
%%capture
# Install cell taken from the official Unsloth notebooks (pins included).
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2
!pip install ollama

### 2. Configuration

In [ ]:
BASE_MODEL = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"
DATASET = "vicgalle/alpaca-gpt4"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 4
MAX_STEPS = 60                # Quick demo run. Raise it (or use epochs) for a real fine-tune.
GGUF_QUANTIZATION = "q4_k_m"  # ~5 GB file. Use "q8_0" for higher quality (~8.5 GB).
OLLAMA_MODEL_NAME = "deepseek_finetuned_model"

### 3. Load the model and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

assert torch.cuda.is_available(), "No CUDA GPU found. In Colab: Runtime > Change runtime type > T4 GPU."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # Auto-detect
    load_in_4bit = True,   # 4-bit quantization so the 8B model fits in a T4
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

### 4. Data prep

The Alpaca dataset has `instruction`, `input` and `output` columns. `to_sharegpt` merges
`instruction` and the optional `input` into a single user prompt, and `conversation_extension`
stitches several rows together to simulate multi-turn conversations.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET, split = "train")
print(dataset.column_names)
dataset[0]

In [ ]:
from unsloth import to_sharegpt, standardize_sharegpt

dataset = to_sharegpt(
    dataset,
    merged_prompt = "{instruction}[[\nYour input is:\n{input}]]",
    output_column_name = "output",
    conversation_extension = 3,  # Select more to handle longer conversations
)
dataset = standardize_sharegpt(dataset)
dataset[0]["conversations"]

### 5. Chat template

The template must contain `{INPUT}` and `{OUTPUT}`. Unsloth also derives the Ollama `Modelfile`
from it, so training and inference use the same prompt format.

In [ ]:
from unsloth import apply_chat_template

chat_template = """Below are some instructions that describe some tasks. Write responses that appropriately complete each request.

### Instruction:
{INPUT}

### Response:
{OUTPUT}"""

dataset = apply_chat_template(
    dataset,
    tokenizer = tokenizer,
    chat_template = chat_template,
    # default_system_message = "You are a helpful assistant",  # Optional
)

### 6. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",  # Use "wandb" etc. for experiment tracking
    ),
)

In [ ]:
trainer_stats = trainer.train()
print(f"Training took {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")

### 7. Export to GGUF

This merges the LoRA adapters into the base model, converts it with llama.cpp and writes an
Ollama `Modelfile` next to the GGUF. It takes several minutes.

In [ ]:
model.save_pretrained_gguf("model", tokenizer, quantization_method = GGUF_QUANTIZATION)

In [ ]:
from pathlib import Path

# The export directory name has changed between Unsloth versions, so locate the Modelfile.
modelfiles = [p for p in Path(".").rglob("Modelfile") if "llama.cpp" not in p.parts]
assert modelfiles, "No Modelfile found. Check the output of the GGUF export cell above."
MODELFILE = modelfiles[0]
print(MODELFILE.read_text())

### 8. Create the Ollama model

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Give the server time to start

In [ ]:
!ollama create {OLLAMA_MODEL_NAME} -f {MODELFILE}

### 9. Chat with the fine-tuned model

In [ ]:
import ollama

response = ollama.chat(
    model = OLLAMA_MODEL_NAME,
    messages = [{"role": "user", "content": "Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8,"}],
)
print(response.message.content)

In [ ]:
from IPython.display import Markdown

response = ollama.chat(
    model = OLLAMA_MODEL_NAME,
    messages = [{"role": "user", "content": "How to add chart to a document?"}],
)
Markdown(response.message.content)

### Using the model on your own machine

Download the `.gguf` file and the `Modelfile` from the export directory, keep them in the same
folder, fix the `FROM` line of the `Modelfile` to point at the local `.gguf`, then run:

```bash
ollama create deepseek_finetuned_model -f Modelfile
ollama run deepseek_finetuned_model
```